# 03 — Baseline Preprocessing

## Objective

The goal of this notebook is to build a clean, reproducible, and leakage-safe baseline preprocessing pipeline for hospital readmission prediction.

Before applying any preprocessing or modeling, we first validate the cleaned dataset produced by Notebook 02.

### Project Principles

- `random_state = 42`
- The test set must not be used for any preprocessing or optimization decision.
- Any learned preprocessing step must be fitted only on the training data.
- Feature engineering and feature selection are not performed in this notebook.
- Hyperparameter optimization is not performed in this notebook.
- The baseline must remain as simple as possible so that future improvements can be measured scientifically.

### Important Data Consideration

The dataset contains multiple encounters for some patients. Therefore:

- `patient_nbr` is treated as a patient identifier, not a regular predictive feature.
- `encounter_id` is an encounter identifier and should not be used as a regular predictive feature.
- The train/validation/test strategy must be designed carefully to avoid patient-level data leakage.

### Current Step

At this stage, we only validate the cleaned dataset from Notebook 02.

No encoding, scaling, feature engineering, feature selection, or model training is performed yet.

In [27]:
# ============================================================
# Notebook 03 — Baseline Preprocessing
# Cell 1 — Validate Notebook 02 Output
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().parent
CLEANED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_data.csv"

print("Project root:")
print(PROJECT_ROOT)

print("\nCleaned dataset path:")
print(CLEANED_DATA_PATH)

# ------------------------------------------------------------
# Check file existence
# ------------------------------------------------------------

if not CLEANED_DATA_PATH.exists():
    raise FileNotFoundError(
        f"\nCleaned dataset was not found at:\n"
        f"{CLEANED_DATA_PATH}\n\n"
        f"Please verify the project structure and Notebook 02 output."
    )

# ------------------------------------------------------------
# Load cleaned dataset
# ------------------------------------------------------------

df = pd.read_csv(CLEANED_DATA_PATH)

# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATASET VALIDATION")
print("=" * 60)

print(f"Shape: {df.shape}")

print("\nTarget column exists:")
print("readmitted" in df.columns)

print("\nData types:")
print(df.dtypes.value_counts())

print("\nExact duplicate rows:")
print(df.duplicated().sum())

print("\nTotal missing values:")
print(df.isna().sum().sum())

# ------------------------------------------------------------
# Constant features
# ------------------------------------------------------------

constant_features = [
    column
    for column in df.columns
    if df[column].nunique(dropna=False) <= 1
]

print("\nRemaining constant features:")
print(constant_features)

# ------------------------------------------------------------
# Target validation
# ------------------------------------------------------------

if "readmitted" not in df.columns:
    raise ValueError("Target column 'readmitted' is missing.")

print("\nTarget distribution:")
print(df["readmitted"].value_counts(dropna=False))

print("\nTarget missing values:")
print(df["readmitted"].isna().sum())

# ------------------------------------------------------------
# Identifier validation
# ------------------------------------------------------------

identifier_columns = [
    column
    for column in ["encounter_id", "patient_nbr"]
    if column in df.columns
]

print("\nIdentifier columns found:")
print(identifier_columns)

for column in identifier_columns:
    print(
        f"{column}: "
        f"{df[column].nunique(dropna=True):,} unique values "
        f"out of {len(df):,} rows"
    )

# ------------------------------------------------------------
# Validation summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Missing values: {df.isna().sum().sum():,}")
print(f"Duplicate rows: {df.duplicated().sum():,}")
print(f"Constant features: {len(constant_features)}")
print(f"Target classes: {df['readmitted'].nunique(dropna=True)}")

print("\nNotebook 02 output loaded successfully.")
print("No preprocessing or modeling has been performed.")

Project root:
e:\all projects\Machine Learning\Advanced-ML-Hospital-Readmission

Cleaned dataset path:
e:\all projects\Machine Learning\Advanced-ML-Hospital-Readmission\data\processed\cleaned_data.csv


C:\Users\MINA\AppData\Local\Temp\ipykernel_24292\2112362865.py:40: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CLEANED_DATA_PATH)



DATASET VALIDATION
Shape: (101766, 47)

Target column exists:
True

Data types:
object    34
int64     13
Name: count, dtype: int64

Exact duplicate rows:
0

Total missing values:
275451

Remaining constant features:
[]

Target distribution:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

Target missing values:
0

Identifier columns found:
['encounter_id', 'patient_nbr']
encounter_id: 101,766 unique values out of 101,766 rows
patient_nbr: 71,518 unique values out of 101,766 rows

VALIDATION SUMMARY
Rows: 101,766
Columns: 47
Missing values: 275,451
Duplicate rows: 0
Constant features: 0
Target classes: 3

Notebook 02 output loaded successfully.
No preprocessing or modeling has been performed.


## Identifier and Target Analysis

Before splitting the data, the patient and encounter identifiers are examined to assess potential data leakage.

The target distribution is also reviewed before defining the baseline preprocessing strategy.

In [28]:
# Identifier analysis

for column in ["encounter_id", "patient_nbr"]:
    print(f"\n{column}")
    print("-" * 40)
    print("Data type:", df[column].dtype)
    print("Missing values:", df[column].isna().sum())
    print("Unique values:", df[column].nunique())
    print("Duplicated values:", df[column].duplicated().sum())


# Number of encounters per patient

encounters_per_patient = df.groupby("patient_nbr").size()

print("\n" + "=" * 50)
print("Patient encounter statistics")
print("=" * 50)

print("Unique patients:", encounters_per_patient.size)
print(
    "Patients with multiple encounters:",
    (encounters_per_patient > 1).sum()
)
print("Maximum encounters per patient:", encounters_per_patient.max())

print("\nDescriptive statistics:")
print(encounters_per_patient.describe())


# Target distribution

print("\n" + "=" * 50)
print("Target distribution")
print("=" * 50)

target_summary = (
    df["readmitted"]
    .value_counts(dropna=False)
    .rename("count")
    .to_frame()
)

target_summary["percentage"] = (
    target_summary["count"] / len(df) * 100
).round(2)

print(target_summary)


encounter_id
----------------------------------------
Data type: int64
Missing values: 0
Unique values: 101766
Duplicated values: 0

patient_nbr
----------------------------------------
Data type: int64
Missing values: 0
Unique values: 71518
Duplicated values: 30248

Patient encounter statistics
Unique patients: 71518
Patients with multiple encounters: 16773
Maximum encounters per patient: 40

Descriptive statistics:
count    71518.000000
mean         1.422942
std          1.090740
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         40.000000
dtype: float64

Target distribution
            count  percentage
readmitted                   
NO          54864       53.91
>30         35545       34.93
<30         11357       11.16


## Remove Non-Predictable Encounters (Expired / Hospice)

`discharge_disposition_id` encodes where the patient went after the encounter. Codes 11, 13, 14, 19, 20, and 21 correspond to the patient having died or been discharged to hospice care during or as a result of this encounter.

These encounters cannot lead to a genuine readmission — the patient did not survive to be readmitted, or their care goal shifted to end-of-life care. Keeping them in the dataset introduces label noise: the model would learn to associate an administrative discharge code with "no readmission," which reflects patient mortality rather than any clinical readmission risk.

These rows are removed before the target is finalized and before splitting, since this is a decision about what the target variable is even measuring.

In [29]:
# discharge_disposition_id codes indicating death or hospice
# Source: UCI IDs_mapping.csv
EXPIRED_HOSPICE_CODES = [11, 13, 14, 19, 20, 21]

rows_before = len(df)

expired_hospice_mask = df["discharge_disposition_id"].isin(EXPIRED_HOSPICE_CODES)

print("Encounters flagged as expired/hospice:", expired_hospice_mask.sum())
print("Breakdown by code:")
print(df.loc[expired_hospice_mask, "discharge_disposition_id"].value_counts().sort_index())

print("\nTarget distribution within flagged encounters:")
print(df.loc[expired_hospice_mask, "readmitted"].value_counts())

df = df.loc[~expired_hospice_mask].reset_index(drop=True)

rows_after = len(df)

print(f"\nRows before: {rows_before:,}")
print(f"Rows after:  {rows_after:,}")
print(f"Rows removed: {rows_before - rows_after:,} ({(rows_before - rows_after) / rows_before:.2%})")

print("\nUpdated target distribution:")
print(df["readmitted"].value_counts())
print(df["readmitted"].value_counts(normalize=True).mul(100).round(2))

Encounters flagged as expired/hospice: 2423
Breakdown by code:
discharge_disposition_id
11    1642
13     399
14     372
19       8
20       2
Name: count, dtype: int64

Target distribution within flagged encounters:
readmitted
NO     2337
<30      43
>30      43
Name: count, dtype: int64

Rows before: 101,766
Rows after:  99,343
Rows removed: 2,423 (2.38%)

Updated target distribution:
readmitted
NO     52527
>30    35502
<30    11314
Name: count, dtype: int64
readmitted
NO     52.87
>30    35.74
<30    11.39
Name: proportion, dtype: float64


## Target Preparation

The original target contains three readmission categories.

For this project, `NO` represents no readmission within 30 days, while both `<30` and `>30` are treated as readmission cases.

In [30]:
# Convert the target to binary

df["readmitted"] = df["readmitted"].map({
    "NO": 0,
    "<30": 1,
    ">30": 1
})

print(df["readmitted"].value_counts().sort_index())

print("\nTarget distribution:")
print(
    df["readmitted"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)


readmitted
0    52527
1    46816
Name: count, dtype: int64

Target distribution:
readmitted
0    52.87
1    47.13
Name: proportion, dtype: float64


## Train, Validation and Test Split

The data is split at the patient level to prevent encounters from the same patient from appearing in different datasets.

The test set is kept separate and will only be used for final evaluation.

In [31]:
from sklearn.model_selection import train_test_split

# Keep patient identifiers separate from model features

patients = df["patient_nbr"].unique()

train_patients, test_patients = train_test_split(
    patients,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_patients, val_patients = train_test_split(
    train_patients,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_df = df[df["patient_nbr"].isin(train_patients)].copy()
val_df = df[df["patient_nbr"].isin(val_patients)].copy()
test_df = df[df["patient_nbr"].isin(test_patients)].copy()

print("Rows:")
print(f"Train: {len(train_df):,}")
print(f"Validation: {len(val_df):,}")
print(f"Test: {len(test_df):,}")

print("\nPatients:")
print(f"Train: {train_df['patient_nbr'].nunique():,}")
print(f"Validation: {val_df['patient_nbr'].nunique():,}")
print(f"Test: {test_df['patient_nbr'].nunique():,}")

print("\nPatient overlap:")

print(
    "Train ∩ Validation:",
    len(set(train_df["patient_nbr"]) & set(val_df["patient_nbr"]))
)

print(
    "Train ∩ Test:",
    len(set(train_df["patient_nbr"]) & set(test_df["patient_nbr"]))
)

print(
    "Validation ∩ Test:",
    len(set(val_df["patient_nbr"]) & set(test_df["patient_nbr"]))
)

Rows:
Train: 63,444
Validation: 15,775
Test: 20,124

Patients:
Train: 44,793
Validation: 11,199
Test: 13,998

Patient overlap:
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


# Save the patient-level split so every future notebook (feature engineering,
# feature selection, optimization) reuses the exact same train/val/test patients.
# This guarantees all before/after comparisons in reports/ are apples-to-apples.

import joblib

split_patients = {
    "train_patients": train_patients,
    "val_patients": val_patients,
    "test_patients": test_patients,
}

joblib.dump(split_patients, MODELS_DIR / "patient_split.pkl")

print("Patient split saved for reuse across all future notebooks:")
print(f"Train patients: {len(train_patients):,}")
print(f"Validation patients: {len(val_patients):,}")
print(f"Test patients: {len(test_patients):,}")

In [32]:
import joblib

split_patients = {
    "train_patients": train_patients,
    "val_patients": val_patients,
    "test_patients": test_patients,
}

joblib.dump(split_patients, models_dir / "patient_split.pkl")

print("Patient split saved for reuse across all future notebooks:")
print(f"Train patients: {len(train_patients):,}")
print(f"Validation patients: {len(val_patients):,}")
print(f"Test patients: {len(test_patients):,}")

Patient split saved for reuse across all future notebooks:
Train patients: 44,793
Validation patients: 11,199
Test patients: 13,998


## Feature and Target Separation

The patient and encounter identifiers are excluded from the predictive features.

The target is separated from the input features before building the preprocessing pipeline.

In [33]:
# Separate target and identifiers from the predictive features

target_column = "readmitted"
identifier_columns = ["encounter_id", "patient_nbr"]

feature_columns = [
    column
    for column in train_df.columns
    if column not in identifier_columns + [target_column]
]

X_train = train_df[feature_columns].copy()
y_train = train_df[target_column].copy()

X_val = val_df[feature_columns].copy()
y_val = val_df[target_column].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[target_column].copy()

print("Feature matrix shapes:")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nTarget shapes:")
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

print("\nNumber of features:", len(feature_columns))

print("\nExcluded identifiers:")
print(identifier_columns)

Feature matrix shapes:
X_train: (63444, 44)
X_val: (15775, 44)
X_test: (20124, 44)

Target shapes:
y_train: (63444,)
y_val: (15775,)
y_test: (20124,)

Number of features: 44

Excluded identifiers:
['encounter_id', 'patient_nbr']


## Feature Types

The feature types are inspected from the training set to define the baseline preprocessing groups.

In [34]:
# Identify numerical and categorical features

numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical features:", len(numerical_features))
print(numerical_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

print("\nTotal features:", len(numerical_features) + len(categorical_features))

Numerical features: 11
['admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Categorical features: 33
['race', 'gender', 'age', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']

Total features: 44


## Missing Values

Missing values are reviewed in the training set before defining the preprocessing pipeline.

In [35]:
missing_summary = (
    X_train.isna()
    .sum()
    .to_frame("missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"]
    .div(len(X_train))
    .mul(100)
    .round(2)
)

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
].sort_values(
    "missing_percentage",
    ascending=False
)

print(missing_summary)

                   missing_count  missing_percentage
max_glu_serum              60197               94.88
A1Cresult                  52751               83.15
medical_specialty          31102               49.02
payer_code                 25166               39.67
race                        1413                2.23
diag_3                       922                1.45
diag_2                       206                0.32
diag_1                        12                0.02
gender                         3                0.00


## Baseline Preprocessing

Numerical features will be imputed with the median and standardized.

Categorical features will be imputed with the most frequent value and one-hot encoded.

All preprocessing steps will be fitted on the training data only.

In [36]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numerical_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]
)

## Apply Preprocessing

The preprocessing pipeline is fitted on the training data and then applied to the validation and test sets.

In [37]:
preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed shapes:")
print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

Processed shapes:
Train: (63444, 2304)
Validation: (15775, 2304)
Test: (20124, 2304)


## Preprocessing Validation

The transformed data is checked to ensure that missing values have been handled and the same feature space is used across all splits.

In [38]:
import numpy as np

print("Output type:", type(X_train_processed))
print("Output shape:", X_train_processed.shape)

print("\nMissing values after preprocessing:")
print("Train:", np.isnan(X_train_processed.data).sum())
print("Validation:", np.isnan(X_val_processed.data).sum())
print("Test:", np.isnan(X_test_processed.data).sum())

print("\nFeature dimensions:")
print("Train:", X_train_processed.shape[1])
print("Validation:", X_val_processed.shape[1])
print("Test:", X_test_processed.shape[1])

Output type: <class 'scipy.sparse._csr.csr_matrix'>
Output shape: (63444, 2304)

Missing values after preprocessing:
Train: 0
Validation: 0
Test: 0

Feature dimensions:
Train: 2304
Validation: 2304
Test: 2304


## Processed Feature Space

The final feature names are extracted from the fitted preprocessing pipeline for inspection and later use in the modeling stage.

In [39]:
processed_feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(processed_feature_names))

print("\nFirst 20 features:")
for feature in processed_feature_names[:20]:
    print(feature)

print("\nLast 20 features:")
for feature in processed_feature_names[-20:]:
    print(feature)

Number of processed features: 2304

First 20 features:
numeric__admission_type_id
numeric__discharge_disposition_id
numeric__admission_source_id
numeric__time_in_hospital
numeric__num_lab_procedures
numeric__num_procedures
numeric__num_medications
numeric__number_outpatient
numeric__number_emergency
numeric__number_inpatient
numeric__number_diagnoses
categorical__race_AfricanAmerican
categorical__race_Asian
categorical__race_Caucasian
categorical__race_Hispanic
categorical__race_Other
categorical__gender_Female
categorical__gender_Male
categorical__age_[0-10)
categorical__age_[10-20)

Last 20 features:
categorical__insulin_Down
categorical__insulin_No
categorical__insulin_Steady
categorical__insulin_Up
categorical__glyburide-metformin_Down
categorical__glyburide-metformin_No
categorical__glyburide-metformin_Steady
categorical__glyburide-metformin_Up
categorical__glipizide-metformin_No
categorical__glipizide-metformin_Steady
categorical__glimepiride-pioglitazone_No
categorical__glimepir

## Save Preprocessing Objects

The fitted preprocessing pipeline and processed feature names are saved for reuse in the modeling stage.

In [40]:
import joblib
from pathlib import Path

models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    preprocessor,
    models_dir / "baseline_preprocessor.pkl"
)

joblib.dump(
    processed_feature_names,
    models_dir / "baseline_feature_names.pkl"
)

print("Preprocessing objects saved successfully.")
print("Saved files:")
print("-", models_dir / "baseline_preprocessor.pkl")
print("-", models_dir / "baseline_feature_names.pkl")

Preprocessing objects saved successfully.
Saved files:
- e:\all projects\Machine Learning\Advanced-ML-Hospital-Readmission\models\baseline_preprocessor.pkl
- e:\all projects\Machine Learning\Advanced-ML-Hospital-Readmission\models\baseline_feature_names.pkl


## Final Validation

The saved preprocessing objects are reloaded to verify that the baseline preprocessing setup is reproducible.

In [41]:
import joblib

saved_preprocessor = joblib.load(
    models_dir / "baseline_preprocessor.pkl"
)

saved_feature_names = joblib.load(
    models_dir / "baseline_feature_names.pkl"
)

print("Preprocessor loaded:", type(saved_preprocessor).__name__)
print("Feature names loaded:", len(saved_feature_names))

print("\nFeature count matches:",
      len(saved_feature_names) == X_train_processed.shape[1])

print("Preprocessor feature count matches:",
      saved_preprocessor.transform(X_train).shape[1]
      == X_train_processed.shape[1])

Preprocessor loaded: ColumnTransformer
Feature names loaded: 2304

Feature count matches: True
Preprocessor feature count matches: True


## Save Baseline Data

The final train, validation, and test sets are saved after preprocessing so they can be reused directly in the modeling stage.

In [42]:
baseline_data = {
    "X_train": X_train_processed,
    "X_val": X_val_processed,
    "X_test": X_test_processed,
    "y_train": y_train,
    "y_val": y_val,
    "y_test": y_test
}

joblib.dump(
    baseline_data,
    models_dir / "baseline_data.pkl"
)

print("Baseline data saved successfully.")
print(models_dir / "baseline_data.pkl")

Baseline data saved successfully.
e:\all projects\Machine Learning\Advanced-ML-Hospital-Readmission\models\baseline_data.pkl


## Summary

In this notebook, the cleaned dataset from Notebook 02 was validated and prepared for baseline modeling.

Encounters with a `discharge_disposition_id` indicating death or hospice transfer (codes 11, 13, 14, 19, 20) were removed prior to target definition and splitting, since these patients cannot be meaningfully readmitted and their inclusion would introduce label noise. This removed 2,423 encounters (2.38% of the data).

The target variable was then converted from three classes into a binary target: `NO → 0` and `<30 / >30 → 1`.

To prevent patient-level data leakage, the dataset was split into training, validation, and test sets based on `patient_nbr`. No patient appeared in more than one split.

The identifier columns `encounter_id` and `patient_nbr` were excluded from the model features, leaving 44 predictive features (11 numerical, 33 categorical). Missing numerical values were handled using median imputation, while categorical missing values were filled using the most frequent category. Numerical features were standardized and categorical features were one-hot encoded. The preprocessing pipeline was fitted only on the training set and then applied to the validation and test sets.

### Final Results

- Train: 63,444 samples (44,793 patients)
- Validation: 15,775 samples (11,199 patients)
- Test: 20,124 samples (13,998 patients)
- Encounters removed (expired/hospice): 2,423
- Predictive features before encoding: 44
- Features after preprocessing: 2,304
- Missing values after preprocessing: 0
- Patient overlap between splits: 0

The fitted preprocessing pipeline and processed feature names were saved for use in the baseline modeling stage.